<a href="https://colab.research.google.com/github/yn99rbccg7-web/Devk-ios/blob/main/colab_finetune_qwen3_4b_redteam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cyberdeck Brain Fine-Tune: Qwen3-4B Abliterated — 5-Stage Continuous Loop

Five domain stages (code → red-team → vuln-research → offensive-net → integration), 500 steps each. Every stage checkpoints its LoRA adapter to Drive; if Colab disconnects, re-run the notebook and the controller resumes at the next incomplete stage. Set `FORCE_RESTART = True` in CONFIG to start a fresh cycle (e.g. monthly, after the training-data recipe is refreshed).

**REDTEAM_REBALANCED 2026-10-10** — red-team specialist recipe: stages 2-4 expanded for offensive methodology (new: Kali network-discovery Q&A in stage 4); generic cyber instruction + redundant MITRE Q&A blocks removed (Trendyol, Tejeswara, MITRE-parquet); replay 15% → 10%; stage 1 (code foundations) kept fully intact. This trains standard pentest/red-team methodology knowledge, not attacks on specific targets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
# Split installs: one package per line so a single failure can't poison the rest.
# (The old combined line died on the xformers pin under Python 3.13, taking bitsandbytes with it.)
!pip install -q bitsandbytes
!pip install -q accelerate
!pip install -q peft
!pip install -q trl
!pip install -q triton
!pip install -q cut_cross_entropy
!pip install -q unsloth_zoo
!pip install -q sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps -q unsloth
print("install done")


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable a T4 GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted")


## CONFIG — the only knobs you should need

- `SUBSET_FRACTION`: fraction of the 115K rows to train on. `0.35` (~40K rows) fits a free-tier session. Set `1.0` only with a bigger GPU / more time.
- `MAX_STEPS`: training steps. 2500 steps x effective batch 8 = 20K samples seen. Raise alongside the subset if you have the hours.

In [ ]:
BASE_MODEL   = "huihui-ai/Huihui-NeoHorse-1-4B-abliterated"
MAX_SEQ_LEN  = 2048
SUBSET_FRACTION = 0.35   # CONFIG: 0.35 of ~135K rows ~= 47K rows, free-tier friendly
MAX_STEPS    = 2500       # CONFIG
SEED = 3407
# "Maximum usable": a free Colab T4 session can meaningfully train ~50K rows
# in MAX_STEPS steps. The loop below caps the training subset here so new
# datasets rebalance the mix instead of growing the run past what fits.
TRAIN_BUDGET_ROWS = 50000

# --- Zero-day / vulnerability-research lane (added 2026-10-01) ---
VR_CVE_SUBSET_N = 12000  # rows taken from the 586K-row jason-oneal CVE set (Apache-2.0, clean)
# License flags: CIRCL and detoxioai/exploit_db publish NO declared license.
# The data itself is public (OSS patch commits / ExploitDB entries), but if you want
# zero license ambiguity, set both to False and train on the Apache-2.0 CVE set only.
USE_CIRCL_PATCH = True   # CIRCL/vulnerability-cwe-patch: vuln -> patch-diff pairs (1-day analysis)
USE_DETOXIO_POC = True   # detoxioai/exploit_db_train_v1: vuln title -> working PoC code

# --- Round-2 lanes: zero-day deepening, offensive net access, red-team ops (added 2026-10-01) ---
PWN_SCENARIOS_N = 11000  # aeby/pwn-scenarios: high-confidence mem-corruption/RCE/CTF-pwn hunt scenarios (CC-BY-4.0)
NUCLEI_TRAIN = True      # OzLabs/ernest-nuclei-templates-v3 split=train: network vuln-template prompts (MIT)
CTFTIME_PWN_N = 5000     # justinwangx/CTFtime: pwn writeup chunks (license undeclared — toggle below)
PENTEST_EXPL_N = 5906    # theelderemo/pentesting-explanations: mitre_attack (2678) + hacktricks (3228) (Apache-2.0)
PENTEST_TOOLS_N = 4474   # oksanany/pentest_tools: pentest tool Q&A (license undeclared — toggle below)
RE_SE_N = 4069           # bshada/reverseengineering.stackexchange.com (StackExchange-derived, treat as CC-BY-SA)
TEJESWARA_N = 3000       # Tejeswara MITRE ATT&CK tactics Q&A, offensive-tactics slice (license undeclared — toggle below)
COWWHYSO_N = 1963        # cowWhySo/pentest-redteam-steering: attack-graph goal->technique (Unlicense)
PRIVESC_N = 1000         # darkknight25/APT_STYLE_Privilege_Escalation_Dataset (MIT)
RTSHELL_N = 1001         # darkknight25/Red_Team_Operations_ShellScript_Dataset (MIT, raw JSONL: load_dataset endpoints broken)
MITRE_QA_N = 475         # ismailtasdelen/mitre-attack-techniques-qa (MIT, raw parquet: rows endpoint 500s)
SARAHWEI_N = 654         # sarahwei/cyber_MITRE_attack_tactics-and-techniques (MIT)
BLOODHOUND_N = 416       # k3nn3dy/bloodhound config=sft: AD attack-path SFT chats (Apache-2.0)
KALI_NET_N = 343  # kali-linux-pentesting-data (verified 343 rows 2026-10-10)
# License toggles: these datasets publish NO declared license (or are StackExchange-derived).
# Default True keeps them in the mix; set any to False for zero license ambiguity.
USE_CTFTIME = True       # justinwangx/CTFtime (undeclared)
USE_OKSANANY = True      # oksanany/pentest_tools (undeclared)
USE_BSHADA_RE = True     # bshada/reverseengineering.stackexchange.com (StackExchange-derived, CC-BY-SA)
USE_TEJESWARA = True     # Tejeswara/cybersec_mitre_attack_tactics_techniques_instruction_data (undeclared)
# Copyleft toggle (default FALSE): CC-BY-SA-4.0 / GPL-3.0 shellcode datasets.
# Flip to True only if you accept copyleft training-data implications for a personal model.
USE_SHELLCODE_COPYLEFT = False  # OSS-forge/Extended_Shellcode_IA32 (CC-BY-SA-4.0) + SoLID/shellcode_i_a32 (GPL-3.0)


# --- Code-execution lane: multilingual SFT for read/write/execute in any language (added 2026-10-01) ---
# All seven datasets below publish permissive licenses (MIT / Apache-2.0 / CC-BY-4.0)
# — no license toggles needed.
MAGICODER_OSS_N = 10000  # ise-uiuc/Magicoder-OSS-Instruct-75K (MIT): problem/solution, stratified by lang
GLAIVE_CODE_N = 12000    # glaiveai/glaive-code-assistant-v3 (Apache-2.0): question/answer
CODEFEEDBACK_N = 8000    # m-a-p/CodeFeedback-Filtered-Instruction (Apache-2.0): query/answer, stratified by lang
MAGICODER_EVOL_N = 8000  # ise-uiuc/Magicoder-Evol-Instruct-110K (Apache-2.0): instruction/response
TOKENBENDER_N = 8000     # TokenBender/code_instructions_122k_alpaca_style (Apache-2.0): instruction/input/output
OPENCODER_N = 8000       # OpenCoder-LLM/opc-sft-stage1 config=realuser_instruct (MIT): instruction/output
QWEN_CODER_N = 6000      # zake7749/Qwen3-Coder-Next-Open-Code-SFT (CC-BY-4.0): problem/response/solution

print("config ok")
# --- 5-stage continuous training loop (added 2026-10-07) ---
# One LoRA adapter, trained sequentially through 5 domain stages.
# Each stage checkpoints to Drive; the train cell auto-resumes incomplete stages.
STAGE_NAMES = {1: "code-foundations", 2: "redteam-ops", 3: "vuln-research",
               4: "offensive-net", 5: "integration-replay"}
N_STAGES = 5
STAGE_STEPS = MAX_STEPS // N_STAGES      # 500 steps per stage
STAGE_ROWS = TRAIN_BUDGET_ROWS // N_STAGES  # ~10K rows per stage
REPLAY_FRAC = 0.10  # stages 2-4 mix 10% replay from earlier stages (anti-forgetting;
                    # mix is now mostly technical, so less replay is needed)
ADAPTER_ROOT = "/content/drive/MyDrive/deck-lora"  # Drive-backed checkpoints
FORCE_RESTART = False  # True: ignore DONE markers, start a fresh 5-stage cycle
print("stage config ok:", STAGE_STEPS, "steps x", N_STAGES, "|", STAGE_ROWS, "rows/stage")
STAGES_THIS_SESSION = None  # e.g. [2]: run ONLY stage 2 this session (one stage per Colab session, in order 1-5)


## 1. Load base model (4-bit) + attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import os

def _latest_adapter():
    done = []
    for n in range(1, 6):
        if os.path.exists(f"{ADAPTER_ROOT}/stage{n}/DONE"):
            done.append(n)
    return f"{ADAPTER_ROOT}/stage{max(done)}" if done else None

_resume = None if FORCE_RESTART else _latest_adapter()
if _resume:
    # Continue training from the last completed stage's LoRA adapter.
    print("resuming LoRA from", _resume)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=_resume,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,          # auto: bf16 on T4
        load_in_4bit=True,   # QLoRA: frozen 4-bit base, trainable adapters
    )
else:
    if FORCE_RESTART:
        print("FORCE_RESTART: fresh adapter from base model")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",  # fits T4 VRAM
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
print("model + LoRA ready")


## 2. Load + format the datasets (red-team/net lanes + vuln-research lane + code-execution lane)

In [ ]:
import base64
from datasets import load_dataset, concatenate_datasets

# --- 5-stage tagging: every dataset is assigned to a training stage ---
# 1: code-foundations | 2: redteam-ops | 3: vuln-research | 4: offensive-net | 5: integration
_STAGE = [1]

def to_text(batch, sys_col=None, user_col="", asst_col=""):
    texts = []
    for i in range(len(batch[user_col])):
        msgs = []
        if sys_col and batch[sys_col][i]:
            msgs.append({"role": "system", "content": str(batch[sys_col][i])})
        msgs.append({"role": "user", "content": str(batch[user_col][i])})
        msgs.append({"role": "assistant", "content": str(batch[asst_col][i])})
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

_STAGE[0] = 3  # stage 3: vuln-research
# 2) CVE -> Metasploit modules: prompt/response
ds_msf = load_dataset("icantiemyshoe/cve-to-metasploit-module", split="train")
ds_msf = ds_msf.map(lambda b: to_text(b, user_col="prompt", asst_col="response"),
                    batched=True, remove_columns=ds_msf.column_names)
print("metasploit:", len(ds_msf))

_STAGE[0] = 4  # stage 4: offensive-net
# 3) CCNA networking Q&A: question/answer
ds_net = load_dataset("Rzkoohi/CCNA_medium", split="train")
ds_net = ds_net.map(lambda b: to_text(b, user_col="question", asst_col="answer"),
                    batched=True, remove_columns=ds_net.column_names)
print("networking:", len(ds_net))

# ---- Vuln-research lane ----

_STAGE[0] = 3  # stage 3: vuln-research
# 4) CIRCL vuln->patch pairs: template the diff into a patch-analysis task.
#    patch_text_b64 must be base64-decoded; rows with no patch are dropped.
def circl_to_text(batch):
    texts = []
    for i in range(len(batch["title"])):
        title = str(batch["title"][i] or "")
        cwes = batch["cwe"][i] or []
        cwe_str = ", ".join(str(c) for c in cwes) if cwes else "unknown"
        patches = batch["patches"][i] or []
        diff, commit_msg = "", ""
        if patches:
            p0 = patches[0]
            commit_msg = str(p0.get("commit_message", "") or "")
            try:
                diff = base64.b64decode(p0.get("patch_text_b64", "") or "").decode("utf-8", errors="replace")
            except Exception:
                diff = ""
        if len(diff) > 6000:
            diff = diff[:6000] + "\n... [diff truncated]"
        user = (
            f"A security patch was committed to fix: {title} (CWE: {cwe_str}).\n"
            "Analyze the diff below: identify the vulnerability class and root cause, "
            "explain how an attacker could have exploited the unpatched code, "
            "and explain how the patch fixes it.\n\n" + diff
        )
        asst = (
            f"Vulnerability: {title} (CWE: {cwe_str}).\n"
            f"Patch commit message: {commit_msg}\n"
            "Analysis: the diff shows the vulnerable code path and the fix. The root cause is the "
            "unsafe condition visible in the removed lines; the patch closes the exploitable state "
            "by enforcing the corrected check, so the vulnerable path is no longer reachable."
        )
        msgs = [{"role": "user", "content": user}, {"role": "assistant", "content": asst}]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

ds_parts = [(3, ds_msf), (4, ds_net)]

if USE_CIRCL_PATCH:
    ds_patch = load_dataset("CIRCL/vulnerability-cwe-patch", split="train")
    ds_patch = ds_patch.filter(lambda r: bool(r["patches"]))  # drop rows with no diff
    ds_patch = ds_patch.map(circl_to_text, batched=True, remove_columns=ds_patch.column_names)
    print("patch-diff:", len(ds_patch))
    ds_parts.append((_STAGE[0], ds_patch))

_STAGE[0] = 3  # stage 3: vuln-research
# 5) ExploitDB PoCs: prompts.prompt -> user, prompts.response -> assistant
def detox_to_text(batch):
    texts = []
    for i in range(len(batch["prompts"])):
        pr = batch["prompts"][i] or {}
        msgs = [
            {"role": "user", "content": str(pr.get("prompt", ""))},
            {"role": "assistant", "content": str(pr.get("response", ""))},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

if USE_DETOXIO_POC:
    ds_poc = load_dataset("detoxioai/exploit_db_train_v1", split="train")
    ds_poc = ds_poc.map(detox_to_text, batched=True, remove_columns=ds_poc.column_names)
    print("exploit-poc:", len(ds_poc))
    ds_parts.append((_STAGE[0], ds_poc))

_STAGE[0] = 3  # stage 3: vuln-research
# 6) CVE analysis Q&A (Apache-2.0): instruction/input -> user, output -> assistant.
#    Only the alpaca config, capped at VR_CVE_SUBSET_N rows.
def alpaca_to_text(batch):
    texts = []
    for i in range(len(batch["instruction"])):
        instr = str(batch["instruction"][i] or "")
        inp = str(batch["input"][i] or "").strip()
        user = instr + ("\n\n" + inp if inp else "")
        msgs = [
            {"role": "user", "content": user},
            {"role": "assistant", "content": str(batch["output"][i] or "")},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

ds_cve = load_dataset(
    "jason-oneal/mitre-stix-cve-exploitdb-dataset-alpaca-chatml-harmony",
    "alpaca", split=f"train[:{VR_CVE_SUBSET_N}]")
ds_cve = ds_cve.map(alpaca_to_text, batched=True, remove_columns=ds_cve.column_names)
print("cve-qa:", len(ds_cve))
ds_parts.append((_STAGE[0], ds_cve))


# ---- Round-2 lanes: zero-day deepening, offensive net access, red-team ops ----
# Each block is self-contained: a failure warns and continues, never kills the run.
import re, json, urllib.request, html as _html
from datasets import Dataset as _ChatDS

def _warn_skip(name, e):
    print(f"  [skip] {name}: {type(e).__name__}: {e}")

def _chat_texts(pairs, name):
    """pairs: iterable of (user, assistant). Chat-template each, skip empties."""
    texts = []
    for u, a in pairs:
        u, a = str(u or "").strip(), str(a or "").strip()
        if not u or not a:
            continue
        msgs = [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    print(f"{name}:", len(texts))
    return texts

def _add_texts(texts, stage=None):
    st = stage if stage is not None else _STAGE[0]
    if texts:
        ds_parts.append((st, _ChatDS.from_list([{"text": t} for t in texts])))

def _strip_html(s):
    s = re.sub(r"<[^>]+>", " ", str(s or ""))
    return re.sub(r"\s+", " ", _html.unescape(s)).strip()

_STAGE[0] = 3  # stage 3: vuln-research
# 7) pwn-scenarios (CC-BY-4.0): high-confidence mem-corruption / RCE / CTF-pwn hunt methodology
try:
    ds = load_dataset("aeby/pwn-scenarios", split="train")
    _MC = ("memory corruption", "buffer overflow", "heap", "stack overflow", "use-after-free",
           "format string", "rop", "rce", "remote code execution", "type confusion",
           "integer overflow", "double free", "ctf", "pwn")
    def _pwn_keep(r):
        prov = r.get("provenance") or {}
        vuln = r.get("vulnerability") or {}
        vc = str(vuln.get("class") or "").lower()
        return prov.get("confidence") == "high" and any(k in vc for k in _MC)
    dsf = ds.filter(_pwn_keep)
    dsf = dsf.select(range(min(PWN_SCENARIOS_N, len(dsf))))
    def _pwn_pair(r):
        vuln = r.get("vulnerability") or {}
        tgt = r.get("target") or {}
        sc = r.get("scenario") or {}
        vc = vuln.get("class") or "vulnerability"
        cwe = vuln.get("cwe") or "unknown"
        tt = tgt.get("type") or "target"
        pre = "; ".join(str(x) for x in (sc.get("preconditions") or []))
        steps = []
        for s in (sc.get("process") or []):
            if isinstance(s, dict):
                tools = ", ".join(str(t) for t in (s.get("tools") or []))
                steps.append(f"{s.get('step','?')}. {s.get('action','')} — {s.get('technique','')}"
                             + (f" [{tools}]" if tools else "")
                             + f" (expect: {s.get('expected_observation','')})")
            else:
                steps.append(f"- {s}")
        impact = "; ".join(str(x) for x in (sc.get("impact") or []))
        rem = "; ".join(str(x) for x in (sc.get("remediation") or []))
        u = f"Hunt {vc} ({cwe}) on a {tt} target. Preconditions: {pre}"
        a = "Methodology:\n" + "\n".join(steps) + f"\nImpact: {impact}\nRemediation: {rem}"
        return u, a
    _add_texts(_chat_texts((_pwn_pair(r) for r in dsf), "pwn-scenarios"))
except Exception as e:
    _warn_skip("pwn-scenarios", e)

_STAGE[0] = 4  # stage 4: offensive-net
# 8) nuclei templates (MIT): network vuln-template prompt -> completion, verbatim
try:
    if NUCLEI_TRAIN:
        ds = load_dataset("OzLabs/ernest-nuclei-templates-v3", split="train")
        _add_texts(_chat_texts(((r.get("prompt"), r.get("completion")) for r in ds), "nuclei"))
except Exception as e:
    _warn_skip("nuclei", e)

_STAGE[0] = 3  # stage 3: vuln-research
# 9) CTFtime pwn writeups (license UNDECLARED — behind USE_CTFTIME)
try:
    if USE_CTFTIME:
        ds = load_dataset("justinwangx/CTFtime", split="train")
        _PWNKW = ("stack", "heap", "rop", "uaf", "format string", "pwn", "shellcode",
                  "buffer overflow", "ret2", "gadget")
        _BOIL = ("leave a reply", "leave a comment", "subscribe to", "table of contents",
                 "all rights reserved", "posted in")
        pairs = []
        for r in ds:
            raw = str(r.get("text_chunk") or "")
            low = raw.lower()
            if not any(k in low for k in _PWNKW):
                continue
            txt = _strip_html(raw)
            if len(txt) < 300:
                continue
            if any(b in low for b in _BOIL):
                continue
            pairs.append(("Explain how this CTF challenge was solved:\n" + txt[:500], txt))
            if len(pairs) >= CTFTIME_PWN_N:
                break
        _add_texts(_chat_texts(pairs, "ctftime-pwn"))
except Exception as e:
    _warn_skip("ctftime", e)

_STAGE[0] = 2  # stage 2: redteam-ops
# 10) pentesting-explanations (Apache-2.0): mitre_attack + hacktricks configs, prompt -> response
try:
    _expl = []
    for _cfg in ("mitre_attack", "hacktricks"):
        _expl.append(load_dataset("theelderemo/pentesting-explanations", _cfg, split="train"))
    ds = concatenate_datasets(_expl)
    ds = ds.select(range(min(PENTEST_EXPL_N, len(ds))))
    _add_texts(_chat_texts(((r.get("prompt"), r.get("response")) for r in ds), "pentest-expl"))
except Exception as e:
    _warn_skip("pentest-explanations", e)

_STAGE[0] = 2  # stage 2: redteam-ops
# 11) pentest_tools (license UNDECLARED — behind USE_OKSANANY): [name] instruction -> answer
try:
    if USE_OKSANANY:
        ds = load_dataset("oksanany/pentest_tools", split="train")
        ds = ds.select(range(min(PENTEST_TOOLS_N, len(ds))))
        _add_texts(_chat_texts(
            ((f"[{r.get('name')}] {r.get('instruction')}", r.get("answer")) for r in ds),
            "pentest-tools"))
except Exception as e:
    _warn_skip("pentest_tools", e)

_STAGE[0] = 3  # stage 3: vuln-research
# 12) RE StackExchange Q&A (StackExchange-derived, treat as CC-BY-SA — behind USE_BSHADA_RE)
try:
    if USE_BSHADA_RE:
        ds = load_dataset("bshada/reverseengineering.stackexchange.com", split="train")
        ds = ds.select(range(min(RE_SE_N, len(ds))))
        pairs = []
        for r in ds:
            u = (str(r.get("Title") or "") + "\n" + _strip_html(r.get("Body"))).strip()
            a = _strip_html(r.get("Answer"))
            pairs.append((u, a))
        _add_texts(_chat_texts(pairs, "re-stackexchange"))
except Exception as e:
    _warn_skip("re-stackexchange", e)

_STAGE[0] = 2  # stage 2: redteam-ops
# 14) pentest-redteam-steering (Unlicense): attack-graph goal -> technique (demarkdowned)
try:
    ds = load_dataset("cowWhySo/pentest-redteam-steering", split="train")
    ds = ds.select(range(min(COWWHYSO_N, len(ds))))
    def _demd(s):
        return re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", str(s or ""))
    _add_texts(_chat_texts(((r.get("goal"), _demd(r.get("target"))) for r in ds), "redteam-steering"))
except Exception as e:
    _warn_skip("redteam-steering", e)

_STAGE[0] = 2  # stage 2: redteam-ops
# 15) APT-style privilege escalation (MIT): [category] description -> tactics + command
try:
    ds = load_dataset("darkknight25/APT_STYLE_Privilege_Escalation_Dataset", split="train")
    ds = ds.select(range(min(PRIVESC_N, len(ds))))
    def _privesc_pair(r):
        u = f"[{r.get('category')}] How do I exploit this? {r.get('description')}"
        a = f"{r.get('tactics')}\n{r.get('description')}\n{r.get('command')}"
        return u, a
    _add_texts(_chat_texts((_privesc_pair(r) for r in ds), "privesc"))
except Exception as e:
    _warn_skip("privesc", e)

_STAGE[0] = 2  # stage 2: redteam-ops
# 16) Red-team shell scripts (MIT) — raw JSONL download (load_dataset endpoints broken, verified live)
try:
    _RT_URL = ("https://huggingface.co/datasets/darkknight25/"
               "Red_Team_Operations_ShellScript_Dataset/resolve/main/red_team_scripting_dataset.jsonl")
    pairs = []
    _req = urllib.request.Request(_RT_URL, headers={"User-Agent": "deck-finetune/1.0"})
    with urllib.request.urlopen(_req, timeout=60) as _resp:
        for _line in _resp:
            _line = _line.decode("utf-8", errors="replace").strip()
            if not _line:
                continue
            try:
                r = json.loads(_line)
            except Exception:
                continue
            u = str(r.get("description") or "").strip()
            scr = str(r.get("script") or "").strip()
            if u and scr:
                pairs.append((u, scr + "\n# usage: " + str(r.get("usage") or "").strip()))
            if len(pairs) >= RTSHELL_N:
                break
    _add_texts(_chat_texts(pairs, "redteam-shell"))
except Exception as e:
    _warn_skip("redteam-shell", e)

_STAGE[0] = 2  # stage 2: redteam-ops
# 18) sarahwei MITRE ATT&CK tactics Q&A (MIT): question -> answer
try:
    ds = load_dataset("sarahwei/cyber_MITRE_attack_tactics-and-techniques", split="train")
    ds = ds.select(range(min(SARAHWEI_N, len(ds))))
    _add_texts(_chat_texts(((r.get("question"), r.get("answer")) for r in ds), "mitre-tactics"))
except Exception as e:
    _warn_skip("mitre-tactics", e)

_STAGE[0] = 4  # stage 4: offensive-net
# 19) bloodhound AD attack paths (Apache-2.0): messages already chat-shaped — drop system turns
try:
    ds = load_dataset("k3nn3dy/bloodhound", "sft", split="train")
    ds = ds.select(range(min(BLOODHOUND_N, len(ds))))
    texts = []
    for r in ds:
        msgs = [m for m in (r.get("messages") or []) if m.get("role") in ("user", "assistant")]
        if len(msgs) < 2:
            continue
        clean = [{"role": m["role"], "content": str(m.get("content") or "")} for m in msgs]
        if not clean[0]["content"].strip() or not clean[-1]["content"].strip():
            continue
        texts.append(tokenizer.apply_chat_template(clean, tokenize=False, add_generation_prompt=False))
    print("bloodhound:", len(texts))
    _add_texts(texts)
except Exception as e:
    _warn_skip("bloodhound", e)

_STAGE[0] = 5  # stage 5: integration-replay
# 20) shellcode intent -> assembly (COPYLEFT — behind USE_SHELLCODE_COPYLEFT, default False)
try:
    if USE_SHELLCODE_COPYLEFT:
        pairs = []
        for r in load_dataset("OSS-forge/Extended_Shellcode_IA32", split="train"):
            pairs.append((r.get("INTENTS"), r.get("SNIPPETS")))
        for r in load_dataset("SoLID/shellcode_i_a32", split="train"):
            pairs.append((r.get("intent"), r.get("snippet")))
        _add_texts(_chat_texts(pairs, "shellcode"))
    else:
        print("shellcode: skipped (USE_SHELLCODE_COPYLEFT=False)")
except Exception as e:
    _warn_skip("shellcode", e)

# ---- Code-execution lane: read/write/execute any language (multilingual SFT) ----
# Stratified helper: round-robin across `lang` values so no single language dominates.
import random as _srand
def _strat_lang_idx(ds, col, n, seed=SEED):
    groups = {}
    for i, r in enumerate(ds):
        groups.setdefault(str(r.get(col) or "other"), []).append(i)
    rng = _srand.Random(seed)
    pools = {g: list(v) for g, v in groups.items()}
    langs = sorted(pools)
    idxs = []
    while len(idxs) < n and langs:
        rng.shuffle(langs)
        for g in list(langs):
            if not pools[g]:
                langs.remove(g); continue
            idxs.append(pools[g].pop(0))
            if len(idxs) >= n:
                break
    rng.shuffle(idxs)
    return idxs

_STAGE[0] = 1  # stage 1: code-foundations
# 21) Magicoder-OSS (MIT): problem -> user, solution -> assistant, stratified by lang
try:
    ds = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
    _idx = _strat_lang_idx(ds, "lang", MAGICODER_OSS_N)
    _add_texts(_chat_texts(((ds[i]["problem"], ds[i]["solution"]) for i in _idx), "magicoder-oss"))
except Exception as e:
    _warn_skip("magicoder-oss", e)

_STAGE[0] = 1  # stage 1: code-foundations
# 22) glaive code assistant v3 (Apache-2.0): question -> user, answer -> assistant
try:
    ds = load_dataset("glaiveai/glaive-code-assistant-v3", split=f"train[:{GLAIVE_CODE_N}]")
    _add_texts(_chat_texts(((r.get("question"), r.get("answer")) for r in ds), "glaive-code"))
except Exception as e:
    _warn_skip("glaive-code", e)

_STAGE[0] = 1  # stage 1: code-foundations
# 23) CodeFeedback filtered (Apache-2.0): query -> user, answer -> assistant, stratified by lang
try:
    ds = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train")
    _idx = _strat_lang_idx(ds, "lang", CODEFEEDBACK_N)
    _add_texts(_chat_texts(((ds[i]["query"], ds[i]["answer"]) for i in _idx), "codefeedback"))
except Exception as e:
    _warn_skip("codefeedback", e)

_STAGE[0] = 1  # stage 1: code-foundations
# 24) Magicoder-Evol (Apache-2.0): instruction -> user, response -> assistant
try:
    ds = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split=f"train[:{MAGICODER_EVOL_N}]")
    _add_texts(_chat_texts(((r.get("instruction"), r.get("response")) for r in ds), "magicoder-evol"))
except Exception as e:
    _warn_skip("magicoder-evol", e)

_STAGE[0] = 1  # stage 1: code-foundations
# 25) TokenBender alpaca-style (Apache-2.0): instruction + input -> user, output -> assistant
try:
    ds = load_dataset("TokenBender/code_instructions_122k_alpaca_style", split=f"train[:{TOKENBENDER_N}]")
    def _tb_pair(r):
        instr = str(r.get("instruction") or "").strip()
        inp = str(r.get("input") or "").strip()
        return instr + ("\n\n" + inp if inp else ""), r.get("output")
    _add_texts(_chat_texts((_tb_pair(r) for r in ds), "tokenbender"))
except Exception as e:
    _warn_skip("tokenbender", e)

_STAGE[0] = 1  # stage 1: code-foundations
# 26) OpenCoder realuser_instruct (MIT): instruction -> user, output -> assistant
try:
    ds = load_dataset("OpenCoder-LLM/opc-sft-stage1", "realuser_instruct", split=f"train[:{OPENCODER_N}]")
    _add_texts(_chat_texts(((r.get("instruction"), r.get("output")) for r in ds), "opencoder"))
except Exception as e:
    _warn_skip("opencoder", e)

_STAGE[0] = 1  # stage 1: code-foundations
# 27) Qwen3-Coder-Next open code SFT (CC-BY-4.0): problem -> user, response + solution -> assistant
#     Keep only fully test-passing rows (test_pass_rate == 1.0) when the column exists.
try:
    ds = load_dataset("zake7749/Qwen3-Coder-Next-Open-Code-SFT", split="train")
    if "test_pass_rate" in ds.column_names:
        ds = ds.filter(lambda r: r.get("test_pass_rate") == 1.0)
    ds = ds.select(range(min(QWEN_CODER_N, len(ds))))
    def _qc_pair(r):
        a = str(r.get("response") or "").strip() + "\n\nSolution:\n" + str(r.get("solution") or "").strip()
        return r.get("problem"), a
    _add_texts(_chat_texts((_qc_pair(r) for r in ds), "qwen-coder-next"))
except Exception as e:
    _warn_skip("qwen-coder-next", e)

_STAGE[0] = 4  # stage 4: offensive-net
# 28) Kali Linux pentesting Q&A (instruction/response): network scanning, service
#     enumeration, access-point discovery methodology. Verified 343 rows 2026-10-10.
try:
    ds = load_dataset("suryanshp1/kali-linux-pentesting-data", split="train")
    ds = ds.select(range(min(KALI_NET_N, len(ds))))
    _add_texts(_chat_texts(((r.get("instruction"), r.get("response")) for r in ds), "kali-net"))
except Exception as e:
    _warn_skip("kali-net", e)


# --- 5-stage assembly: group datasets by stage; build per-stage training sets ---
from collections import defaultdict
_stage_groups = defaultdict(list)
for _st, _ds in ds_parts:
    _stage_groups[_st].append(_ds)
for _s in range(1, 6):
    _rows = sum(len(d) for d in _stage_groups.get(_s, []))
    print(f"stage {_s} ({STAGE_NAMES[_s]}): {len(_stage_groups.get(_s, []))} datasets, {_rows:,} rows")

def build_stage(n):
    """Training set for stage n: domain mix + replay of earlier stages (anti-forgetting)."""
    assert 1 <= n <= 5
    dom_list = _stage_groups.get(n, [])
    parts, take = [], 0
    if n <= 4:
        dom_cap = int(STAGE_ROWS * (1 - REPLAY_FRAC)) if n > 1 else STAGE_ROWS
        if dom_list:
            dom = concatenate_datasets(dom_list).shuffle(seed=SEED + n)
            take = min(len(dom), dom_cap)
            parts.append(dom.select(range(take)))
            print(f"  stage {n}: domain rows {take:,}")
        if n > 1:
            prior_list = [d for s in range(1, n) for d in _stage_groups.get(s, [])]
            if prior_list:
                prior = concatenate_datasets(prior_list).shuffle(seed=SEED + 100 + n)
                rep = min(len(prior), STAGE_ROWS - take)
                if rep > 0:
                    parts.append(prior.select(range(rep)))
                    print(f"  stage {n}: replay rows {rep:,}")
    else:
        # Stage 5: integration — replay-heavy consolidation across all prior stages.
        prior_list = [d for s in range(1, 5) for d in _stage_groups.get(s, [])]
        prior = concatenate_datasets(prior_list).shuffle(seed=SEED + 105)
        take = min(len(prior), STAGE_ROWS)
        parts.append(prior.select(range(take)))
        print(f"  stage 5: integration replay rows {take:,}")
        if dom_list:  # shellcode lane, only when the copyleft toggle is on
            extra = concatenate_datasets(dom_list).shuffle(seed=SEED + 5)
            parts.append(extra)
            print(f"  stage 5: extra domain rows {len(extra):,}")
    assert parts, f"stage {n} has no data — check dataset blocks"
    ds = concatenate_datasets(parts).shuffle(seed=SEED + n)
    print(f"stage {n} total: {len(ds):,} rows")
    return ds

print("sample stage-1 row:\n", concatenate_datasets(_stage_groups[1]).shuffle(seed=SEED).select(range(1))[0]["text"][:600])


## 3. Train — 5-stage continuous loop (SFTTrainer)

Runs every incomplete stage in order: build the stage dataset → train 500 steps → checkpoint the LoRA adapter to Drive → next stage. **If Colab disconnects:** just re-run the notebook from the top — the controller detects the `DONE` markers in Drive and resumes at the next incomplete stage. Nothing is re-trained.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os

os.makedirs(ADAPTER_ROOT, exist_ok=True)
def _stage_dir(n):
    return f"{ADAPTER_ROOT}/stage{n}"

# --- controller: pick this session's stages ---
# STAGES_THIS_SESSION = [k] runs exactly one stage per Colab session.
# Recommended: 5 sessions, STAGES_THIS_SESSION = [1], then [2], ... [5], in order.
# None = run all remaining incomplete stages in one go.
def _done(n):
    return os.path.exists(f"{_stage_dir(n)}/DONE") and not FORCE_RESTART

if STAGES_THIS_SESSION:
    planned = [n for n in STAGES_THIS_SESSION if 1 <= n <= 5 and not _done(n)]
    print(f"this session: stages {planned}")
else:
    start_stage = 1
    for _n in range(1, 6):
        if _done(_n):
            start_stage = _n + 1
    planned = list(range(start_stage, 6))
    print(f"resuming at stage {start_stage}/5")

if not planned:
    print("nothing to do — all requested stages already complete")
for _n in planned:
    print(f"\n===== STAGE {_n}/5: {STAGE_NAMES[_n]} =====")
    _ds = build_stage(_n)
    _trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        dataset_num_proc=2,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,   # effective batch = 8
            warmup_steps=20,
            max_steps=STAGE_STEPS,           # 500 per stage; 2500 total
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=25,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=SEED,
            output_dir=f"outputs-stage{_n}",
            save_steps=250,
            save_total_limit=1,
            report_to="none",
        ),
    )
    _trainer.train()
    model.save_pretrained(_stage_dir(_n))       # LoRA adapter -> Drive
    tokenizer.save_pretrained(_stage_dir(_n))
    open(f"{_stage_dir(_n)}/DONE", "w").write("ok")
    print(f"stage {_n} complete — adapter checkpointed to Drive")
print("\nALL 5 STAGES COMPLETE — run the smoke test + GGUF export cells below")


## 4. Quick smoke test before export

Ask it something from the specialty and confirm it answers like a red-teamer, not a generic chatbot.

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "user", "content": "I found CVE-2015-5119 on a target. Which Metasploit module applies and what does a basic check look like?"},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(inputs, max_new_tokens=256, temperature=0.3, min_p=0.1)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 5. Export to GGUF (Q4_K_M) for Ollama

In [ ]:
# Merge LoRA into 16-bit, then quantize straight to GGUF Q4_K_M.
# This takes a while on CPU — let it run.
model.save_pretrained_gguf("deck-redteam-gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF saved to ./deck-redteam-gguf/")
!ls -lh deck-redteam-gguf/

In [ ]:
from google.colab import files
import glob
gguf = glob.glob("deck-redteam-gguf/*.gguf")[0]
print("downloading:", gguf)
files.download(gguf)

## 6. Next step (back in chat)

Send the downloaded `.gguf` file back in chat. I'll load it into Ollama as your new deck brain (replacing the stock 4B), re-run the benchmark + tool tests against it, and report what the fine-tune actually bought you.

**If Colab disconnects mid-run:** just re-run the notebook from the top — completed stages are detected in Drive and skipped, training resumes where it stopped. No knobs need changing.

**Next cycle:** the training-data recipe is refreshed monthly; set `FORCE_RESTART = True` in CONFIG to start a fresh 5-stage cycle on the updated mix.
